# REGRESSION ANALYSIS

This notebook performs regression analysis to understand which marketing metrics best predict the number of purchases.

This stage includes:

* Correlation Analysis
* Simple Linear Regression
* Multiple Linear Regression
* Model Diagnostics (Residual Analysis)
* Feature Importance
* Group Comparison via Regression


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/processed/cleaned_marketing.csv')
df['date'] = pd.to_datetime(df['date'])

# Drop rows with missing values for regression
df_clean = df.dropna().copy()
print(f'Dataset shape (after dropping NaN): {df_clean.shape}')
df_clean.head()

## 1. Correlation Analysis


In [ ]:
numeric_cols = ['spend_usd', 'impressions', 'reach', 'website_clicks',
                'searches', 'view_content', 'add_to_cart', 'purchase']

corr_matrix = df_clean[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True
)
plt.title('Correlation Matrix of Marketing Metrics')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with purchase
purchase_corr = corr_matrix['purchase'].drop('purchase').sort_values(ascending=False)
print('Correlation with Purchase:')
print(purchase_corr)

In [ ]:
plt.figure(figsize=(8, 5))
purchase_corr.plot(kind='bar', color=['steelblue' if v >= 0 else 'salmon' for v in purchase_corr])
plt.axhline(y=0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Purchase')
plt.ylabel('Pearson Correlation')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Simple Linear Regression

Predicting `purchase` from the most correlated single feature.


In [ ]:
best_feature = purchase_corr.abs().idxmax()
print(f'Best predictor: {best_feature} (r = {purchase_corr[best_feature]:.4f})')

X_simple = df_clean[[best_feature]]
y = df_clean['purchase']

# Statsmodels for detailed output
X_sm = sm.add_constant(X_simple)
model_simple = sm.OLS(y, X_sm).fit()
print(model_simple.summary())

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_clean[best_feature], y, alpha=0.6, label='Data')

x_line = np.linspace(df_clean[best_feature].min(), df_clean[best_feature].max(), 100)
y_line = model_simple.params[1] * x_line + model_simple.params[0]
plt.plot(x_line, y_line, color='red', linewidth=2, label='Regression Line')

plt.xlabel(best_feature.replace('_', ' ').title())
plt.ylabel('Purchase')
plt.title(f'Simple Linear Regression: Purchase vs {best_feature}')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Multiple Linear Regression

Using all numeric predictors to model purchases.


In [ ]:
feature_cols = ['spend_usd', 'impressions', 'reach', 'website_clicks',
                'searches', 'view_content', 'add_to_cart']

X = df_clean[feature_cols]
y = df_clean['purchase']

X_sm = sm.add_constant(X)
model_multi = sm.OLS(y, X_sm).fit()
print(model_multi.summary())

In [ ]:
# sklearn for metrics
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f'R² Score:  {r2:.4f}')
print(f'RMSE:      {rmse:.2f}')
print(f'MAE:       {mae:.2f}')

## 4. Coefficient Plot


In [ ]:
# Standardized coefficients for comparison
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_sm = sm.add_constant(X_scaled)

model_std = sm.OLS(y, X_scaled_sm).fit()
coef_df = pd.DataFrame({
    'Feature': ['const'] + feature_cols,
    'Coefficient': model_std.params,
    'P-value': model_std.pvalues
}).query('Feature != "const"')

coef_df['Significant'] = coef_df['P-value'] < 0.05
coef_df = coef_df.sort_values('Coefficient', ascending=True)

plt.figure(figsize=(9, 6))
colors = ['steelblue' if c >= 0 else 'salmon' for c in coef_df['Coefficient']]
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)

# Mark significant features
for i, (_, row) in enumerate(coef_df.iterrows()):
    if row['Significant']:
        plt.text(
            row['Coefficient'] + 0.5,
            i,
            '*',
            va='center',
            fontsize=14,
            color='black'
        )

plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Standardized Coefficient')
plt.title('Standardized Regression Coefficients\n(* = p < 0.05)')
plt.tight_layout()
plt.show()

## 5. Residual Analysis


In [ ]:
y_pred_all = model_multi.predict(sm.add_constant(X))
residuals = y - y_pred_all

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Residuals vs Fitted
axes[0].scatter(y_pred_all, residuals, alpha=0.6)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# QQ Plot
stats.probplot(residuals, plot=axes[1])
axes[1].set_title('QQ Plot of Residuals')

# Residual Distribution
axes[2].hist(residuals, bins=15, edgecolor='black', color='steelblue', alpha=0.7)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

## 6. Regression by Group (Control vs Test)


In [ ]:
df_clean['group_encoded'] = (df_clean['group'] == 'test').astype(int)

feature_with_group = feature_cols + ['group_encoded']
X_grp = df_clean[feature_with_group]
X_grp_sm = sm.add_constant(X_grp)

model_grp = sm.OLS(y, X_grp_sm).fit()
print(model_grp.summary())

In [ ]:
group_coef = model_grp.params['group_encoded']
group_pval = model_grp.pvalues['group_encoded']

print('=== Group Effect on Purchase ===')
print(f'Coefficient of group (test vs control): {group_coef:.4f}')
print(f'P-value: {group_pval:.4f}')
print()
if group_pval < 0.05:
    direction = 'higher' if group_coef > 0 else 'lower'
    print(f'✓ The test group has SIGNIFICANTLY {direction} purchases than control (p < 0.05).')
else:
    print('✗ No significant difference between groups after controlling for other features (p ≥ 0.05).')

In [ ]:
# Scatter of predicted vs actual
y_pred_grp = model_grp.predict(sm.add_constant(X_grp))

plt.figure(figsize=(8, 6))
plt.scatter(
    y_pred_grp[df_clean['group'] == 'control'],
    y[df_clean['group'] == 'control'],
    alpha=0.7, label='Control', color='steelblue'
)
plt.scatter(
    y_pred_grp[df_clean['group'] == 'test'],
    y[df_clean['group'] == 'test'],
    alpha=0.7, label='Test', color='salmon'
)
min_val = min(y_pred_grp.min(), y.min())
max_val = max(y_pred_grp.max(), y.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect Fit')
plt.xlabel('Predicted Purchase')
plt.ylabel('Actual Purchase')
plt.title('Actual vs Predicted Purchase (by Group)')
plt.legend()
plt.tight_layout()
plt.show()

## INTERPRETATION

**Correlation:** `add_to_cart` and `searches` show the strongest correlations with `purchase`, indicating that customers who add items to cart or search more tend to complete more purchases.

**Multiple Regression Model:** The model explains a portion of the variance in purchases. Statistically significant predictors (p < 0.05) are the most important drivers.

**Group Effect:** After controlling for all other marketing metrics, the regression coefficient of the `group` variable tells us whether the test campaign independently drove more purchases beyond what the funnel metrics alone predict.

**Residuals:** Ideally, residuals should be normally distributed and show no pattern against fitted values. Any systematic pattern suggests model misspecification or non-linearity.
